In [0]:
df = spark.read.csv(
    "file:/Workspace/orders.csv",
    header=True,
    inferSchema=True
)

display(df)

order_id,supplier_id,item,quantity,delivery_date,status
1,S1,Laptop,10,2024-01-01,Delivered
2,S2,Mouse,50,2024-01-15,Pending
3,S1,Keyboard,30,2024-01-10,Delayed
4,S3,Monitor,5,2023-12-20,Delivered
5,S2,Webcam,20,2024-02-01,Pending
6,S1,Headphones,15,2023-11-15,Delayed
7,S3,Charger,40,2023-12-01,Delayed
8,S2,USB Hub,25,2024-01-20,Pending
9,S1,SSD,8,2023-10-10,Delayed
10,S3,Printer,3,2024-02-10,Pending


In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- status: string (nullable = true)



In [0]:
# Clean and Filter Data
from pyspark.sql.functions import col, to_date, datediff, current_date, when
# Format date column
df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd"))

In [0]:
# Drop null values
df = df.dropna(subset=["order_id", "delivery_date"])

In [0]:
# Add delay calculation
df = df.withColumn("delay_days", datediff(current_date(), col("delivery_date")))
df = df.withColumn("is_delayed", when(col("delay_days") > 0, 1).otherwise(0))


In [0]:
# Filter only delayed orders
delayed_df = df.filter(col("is_delayed") == 1)
delayed_df.show()


+--------+-----------+----------+--------+-------------+---------+----------+----------+
|order_id|supplier_id|      item|quantity|delivery_date|   status|delay_days|is_delayed|
+--------+-----------+----------+--------+-------------+---------+----------+----------+
|       1|         S1|    Laptop|      10|   2024-01-01|Delivered|       863|         1|
|       2|         S2|     Mouse|      50|   2024-01-15|  Pending|       849|         1|
|       3|         S1|  Keyboard|      30|   2024-01-10|  Delayed|       854|         1|
|       4|         S3|   Monitor|       5|   2023-12-20|Delivered|       875|         1|
|       5|         S2|    Webcam|      20|   2024-02-01|  Pending|       832|         1|
|       6|         S1|Headphones|      15|   2023-11-15|  Delayed|       910|         1|
|       7|         S3|   Charger|      40|   2023-12-01|  Delayed|       894|         1|
|       8|         S2|   USB Hub|      25|   2024-01-20|  Pending|       844|         1|
|       9|         S1

In [0]:
#  SQL Analysis using Spark SQL
df.createOrReplaceTempView("orders")

result = spark.sql("""
    SELECT supplier_id,
           COUNT(*) AS total_orders,
           SUM(is_delayed) AS delayed_orders,
           AVG(delay_days) AS avg_delay
    FROM orders
    GROUP BY supplier_id
    ORDER BY delayed_orders DESC
""")
result.show()

+-----------+------------+--------------+-----------------+
|supplier_id|total_orders|delayed_orders|        avg_delay|
+-----------+------------+--------------+-----------------+
|         S1|           4|             4|           893.25|
|         S3|           3|             3|            864.0|
|         S2|           3|             3|841.6666666666666|
+-----------+------------+--------------+-----------------+



In [0]:
# Save as Delta Table 
df.write.format("delta").mode("overwrite").saveAsTable("cleaned_orders")
print("Saved as Delta Table successfully!")

Saved as Delta Table successfully!


In [0]:
spark.sql("SELECT * FROM cleaned_orders LIMIT 5").show()

+--------+-----------+--------+--------+-------------+---------+----------+----------+
|order_id|supplier_id|    item|quantity|delivery_date|   status|delay_days|is_delayed|
+--------+-----------+--------+--------+-------------+---------+----------+----------+
|       1|         S1|  Laptop|      10|   2024-01-01|Delivered|       863|         1|
|       2|         S2|   Mouse|      50|   2024-01-15|  Pending|       849|         1|
|       3|         S1|Keyboard|      30|   2024-01-10|  Delayed|       854|         1|
|       4|         S3| Monitor|       5|   2023-12-20|Delivered|       875|         1|
|       5|         S2|  Webcam|      20|   2024-02-01|  Pending|       832|         1|
+--------+-----------+--------+--------+-------------+---------+----------+----------+



In [0]:
import pandas as pd

cleaned_orders_pd = spark.sql("SELECT * FROM cleaned_orders").toPandas()
cleaned_orders_pd.to_csv("/Workspace/cleaned_orders_output.csv", index=False)

print("Download Complete!")
print(cleaned_orders_pd)

Download Complete!
   order_id supplier_id        item  ...     status delay_days is_delayed
0         1          S1      Laptop  ...  Delivered        863          1
1         2          S2       Mouse  ...    Pending        849          1
2         3          S1    Keyboard  ...    Delayed        854          1
3         4          S3     Monitor  ...  Delivered        875          1
4         5          S2      Webcam  ...    Pending        832          1
5         6          S1  Headphones  ...    Delayed        910          1
6         7          S3     Charger  ...    Delayed        894          1
7         8          S2     USB Hub  ...    Pending        844          1
8         9          S1         SSD  ...    Delayed        946          1
9        10          S3     Printer  ...    Pending        823          1

[10 rows x 8 columns]
